# Capstone starter: fine-tune → evaluate → optimize (optional)
**LLMs From Basics · Capstone**

The whole capstone pipeline in miniature, on a CPU, in under a minute:

1. a toy **ticket-routing** task (8 teams) and a frozen **eval set**, built first;
2. a small "pretrained" model as the **baseline**;
3. an eval harness with **bootstrap confidence intervals** and **paired** comparisons;
4. a **LoRA** fine-tune with checkpoint selection on a dev split;
5. **int8 weight quantization**, then a final table of accuracy and size.

Each exercise has a `# TODO` cell and a **check** cell with `assert`s. Solutions are at the bottom. When you start your real project, keep the structure and swap in a real model (via Hugging Face `transformers` + `peft`) and a real dataset.

In [ ]:
import copy, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
torch.set_num_threads(1)   # tiny models run fastest on one thread
print("ready")

## 0 · The task, and the eval set first
Tickets are bags of words. Each of 8 teams has its own keywords; a ticket mentions two keywords of its team, one keyword of some other team (to confuse things) and a few filler words. As in real data, about 12% of tickets are ambiguous or mislabelled, so even a perfect model can't reach 100%.

The twist: three product teams (5, 6, 7) were **renamed** since the base model was trained, so today's tickets use **new keywords** for them. The base model has never seen those words. That's our reason to fine-tune.

Note the order: the **eval set is generated and frozen first**, from today's distribution, and never used for training or for choosing checkpoints (that's what the dev split is for).

In [ ]:
N_TEAMS, K, N_FILL = 8, 6, 40
OLD = {t: list(range(t * K, (t + 1) * K)) for t in range(N_TEAMS)}                       # words 0..47
NEW = dict(OLD); NEW.update({t: list(range(48 + (t - 5) * K, 48 + (t - 4) * K)) for t in (5, 6, 7)})  # words 48..65
FILL = list(range(66, 66 + N_FILL))
V = 66 + N_FILL

def make_tickets(n, vocab, seed, noise=0.12):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, V), dtype=np.float32); y = rng.integers(0, N_TEAMS, n)
    for i, t in enumerate(y):
        other = rng.choice([s for s in range(N_TEAMS) if s != t])
        words = list(rng.choice(vocab[t], 2, replace=False)) + [rng.choice(vocab[other])] + list(rng.choice(FILL, 5))
        for w in words:
            X[i, w] += 1
    flip = rng.random(n) < noise                                   # ambiguous / mislabelled tickets
    y[flip] = (y[flip] + rng.integers(1, N_TEAMS, flip.sum())) % N_TEAMS
    return torch.tensor(X), torch.tensor(y)

X_eval, y_eval = make_tickets(400, NEW, seed=2024)     # frozen eval set: built first, never trained on
X_old, y_old = make_tickets(6000, OLD, seed=1)         # what the base model was trained on
X_train, y_train = make_tickets(2000, NEW, seed=2)     # today's training tickets
X_dev, y_dev = make_tickets(300, NEW, seed=3)          # for choosing checkpoints
print("eval", tuple(X_eval.shape), "train", tuple(X_train.shape), "dev", tuple(X_dev.shape))

### The "pretrained" base model
A stand-in for an open-weight LLM: a small MLP trained on the old tickets.

In [ ]:
def make_model():
    return nn.Sequential(nn.Linear(V, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, N_TEAMS))

def train_steps(model, X, y, steps, lr=1e-2, seed=0):
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    for _ in range(steps):
        idx = torch.randint(0, len(X), (128,), generator=gen)
        loss = F.cross_entropy(model(X[idx]), y[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    return model

torch.manual_seed(0)
base = train_steps(make_model(), X_old, y_old, steps=300)

## 1 · Eval harness with a bootstrap CI
`correct_vector(model, X, y)` returns a NumPy array of 0/1, one per eval example. Then write `bootstrap_ci(correct, n_boot, seed)`: resample the examples **with replacement** `n_boot` times, compute the accuracy of each resample, and return `(accuracy, lower, upper)` where the bounds are the 2.5th and 97.5th percentiles.

*(Vectorise it: `rng.integers(0, n, size=(n_boot, n))` gives all the resampled indices at once.)*

In [ ]:
def correct_vector(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(-1) == y).numpy().astype(float)

def bootstrap_ci(correct, n_boot=10_000, seed=0):
    correct = np.asarray(correct, dtype=float)
    rng = np.random.default_rng(seed)
    # TODO: resample, compute each resample's mean, return (mean, 2.5th pct, 97.5th pct)
    raise NotImplementedError("Exercise 1")

**Check 1.** The lesson's worked example: 150 of 200 right gives about 75% [69%, 81%].

In [ ]:
acc, lo, hi = bootstrap_ci(np.r_[np.ones(150), np.zeros(50)])
print(f"{acc:.3f} [{lo:.3f}, {hi:.3f}]")
assert acc == 0.75 and abs(lo - 0.69) < 0.015 and abs(hi - 0.81) < 0.015

base_correct = correct_vector(base, X_eval, y_eval)
acc, lo, hi = bootstrap_ci(base_correct)
print(f"baseline on the frozen eval set: {acc:.1%} [{lo:.1%}, {hi:.1%}]")
assert 0.4 < acc < 0.8, "the base model should be mediocre on today's tickets"
print("✓ exercise 1")

## 2 · Paired comparison
Two models scored on the **same** eval examples should be compared example by example. Write `paired_bootstrap(correct_a, correct_b)`: resample example **indices** (the same indices for both models), and return `(diff, lower, upper)` for the accuracy difference `b − a`.

In [ ]:
def paired_bootstrap(correct_a, correct_b, n_boot=10_000, seed=0):
    a, b = np.asarray(correct_a, float), np.asarray(correct_b, float)
    rng = np.random.default_rng(seed)
    # TODO: resample shared indices; return (mean diff, 2.5th pct, 97.5th pct) of b - a
    raise NotImplementedError("Exercise 2")

**Check 2.** The lesson's example: 200 tickets, 34 only model B gets right, 8 only model A. The formula gave +13 points, CI [+6.9, +19.1].

In [ ]:
a = np.r_[np.zeros(34), np.ones(8), np.ones(116), np.zeros(42)]   # base: 124 right
b = np.r_[np.ones(34), np.zeros(8), np.ones(116), np.zeros(42)]   # LoRA: 150 right
d, lo, hi = paired_bootstrap(a, b)
print(f"{d:+.3f} [{lo:+.3f}, {hi:+.3f}]")
assert abs(d - 0.13) < 1e-9 and abs(lo - 0.069) < 0.012 and abs(hi - 0.191) < 0.012
print("✓ exercise 2")

## 3 · LoRA fine-tune
Here's the LoRA layer from lesson M3.7. Write `add_lora(model, r, alpha)`: copy the model, freeze everything, and wrap every `nn.Linear` in a `LoRALinear`. The provided loop then trains the adapters and keeps the checkpoint with the best **dev** accuracy (never the eval set).

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scaling = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scaling

def merge(model):
    out = []
    for layer in model:
        if isinstance(layer, LoRALinear):
            new = nn.Linear(layer.base.in_features, layer.base.out_features)
            with torch.no_grad():
                new.weight.copy_(layer.base.weight + layer.scaling * layer.B @ layer.A)
                new.bias.copy_(layer.base.bias)
            out.append(new)
        else:
            out.append(copy.deepcopy(layer))
    return nn.Sequential(*out)

In [ ]:
def add_lora(model, r=4, alpha=8):
    model = copy.deepcopy(model)
    # TODO: freeze all parameters; replace each nn.Linear model[i] with LoRALinear(model[i], r, alpha)
    raise NotImplementedError("Exercise 3")
    return model

**Check 3.** Train in chunks of 50 steps and keep the best checkpoint by dev accuracy.

In [ ]:
torch.manual_seed(0)
lora = add_lora(base, r=4, alpha=8)
n_train = sum(p.numel() for p in lora.parameters() if p.requires_grad)
assert n_train == 4 * (V + 64) + 4 * (64 + 64) + 4 * (64 + N_TEAMS)
assert np.array_equal(correct_vector(lora, X_eval, y_eval), base_correct), "a fresh adapter is a no-op"

best, best_dev = None, -1
for chunk in range(8):
    train_steps(lora, X_train, y_train, steps=50, seed=chunk)
    dev = correct_vector(lora, X_dev, y_dev).mean()
    print(f"step {50 * (chunk + 1):3d}  dev acc {dev:.3f}")
    if dev > best_dev:
        best, best_dev = copy.deepcopy(lora), dev
tuned = merge(best)
tuned_correct = correct_vector(tuned, X_eval, y_eval)
acc, lo, hi = bootstrap_ci(tuned_correct)
print(f"fine-tuned on the frozen eval set: {acc:.1%} [{lo:.1%}, {hi:.1%}]")
d, dlo, dhi = paired_bootstrap(base_correct, tuned_correct)
print(f"gain vs baseline: {d:+.1%} [{dlo:+.1%}, {dhi:+.1%}]")
assert dlo > 0, "the gain should be clearly above zero"
print("✓ exercise 3")

## 4 · int8 weight quantization
Quantize each weight matrix **per output row**, symmetrically: for row $i$, $s_i = \max_j |W_{ij}| / 127$, then $q_{ij} = \operatorname{round}(W_{ij}/s_i)$ stored as `int8`. Dequantize with $\hat W_{ij} = q_{ij}\, s_i$. Biases stay in float32.

In [ ]:
def quantize_int8(W):
    # TODO: return (q as torch.int8 tensor, scale as a float32 tensor of shape (rows, 1))
    raise NotImplementedError("Exercise 4a")

def dequantize(q, scale):
    # TODO
    raise NotImplementedError("Exercise 4b")

**Check 4**

In [ ]:
W = torch.randn(64, 106)
q, s = quantize_int8(W)
assert q.dtype == torch.int8 and s.shape == (64, 1)
assert q.abs().max() <= 127
err = (dequantize(q, s) - W).abs()
assert torch.all(err <= s / 2 + 1e-6), "rounding error is at most half a step"

def quantize_model(model):
    # simulated quantization: store int8 + scales, run with dequantized weights
    qm, nbytes = copy.deepcopy(model), 0
    for layer in qm:
        if isinstance(layer, nn.Linear):
            q, s = quantize_int8(layer.weight.data)
            layer.weight.data = dequantize(q, s)
            nbytes += q.numel() * 1 + s.numel() * 4 + layer.bias.numel() * 4
    return qm, nbytes

def fp32_bytes(model):
    return sum(p.numel() * 4 for p in model.parameters())

quant, q_bytes = quantize_model(tuned)
quant_correct = correct_vector(quant, X_eval, y_eval)
print(f"size: fp32 {fp32_bytes(tuned) / 1024:.1f} KB → int8 {q_bytes / 1024:.1f} KB")
assert q_bytes < 0.3 * fp32_bytes(tuned)
print("✓ exercise 4")

## 5 · The report and the decision
All three models on the identical frozen eval set. The decision rule, written **before** looking: *ship the int8 model if the lower end of the 95% CI for its accuracy change (vs the fine-tuned fp32 model) is above −3 points.*

In [ ]:
rows = [("base (few-shot stand-in)", base, base_correct),
        ("LoRA fine-tune (merged)", tuned, tuned_correct),
        ("LoRA + int8", quant, quant_correct)]
print(f"{'model':28s} {'accuracy [95% CI]':24s} {'size':>9s}")
for name, m, c in rows:
    acc, lo, hi = bootstrap_ci(c)
    size = q_bytes if m is quant else fp32_bytes(m)
    print(f"{name:28s} {acc:6.1%} [{lo:5.1%}, {hi:5.1%}]   {size / 1024:6.1f} KB")
d, lo, hi = paired_bootstrap(tuned_correct, quant_correct)
print(f"\nint8 vs fp32: {d:+.1%} [{lo:+.1%}, {hi:+.1%}]")
print("decision:", "ship int8" if lo > -0.03 else "don't ship int8 yet: collect more eval data or keep fp32")

## Experiment
- **Eval-set size.** Re-run the report on only the first 100 eval tickets. How much wider do the CIs get? (They should grow by about $\sqrt{400/100} = 2\times$.)
- **Rank.** Try `r=1` and `r=16`. Does this task need more than rank 1?
- **Fewer bits.** Change `127` to `7` in `quantize_int8` (4-bit levels) and re-run. Does the decision change?
- **Forgetting.** Score `base` and `tuned` on tickets from the *old* distribution (`make_tickets(400, OLD, seed=99)`). What did fine-tuning cost?

In [ ]:
c100 = [c[:100] for _, _, c in rows]
for (name, _, _), c in zip(rows, c100):
    acc, lo, hi = bootstrap_ci(c)
    print(f"{name:28s} n=100: {acc:6.1%} [{lo:5.1%}, {hi:5.1%}]  width {hi - lo:.1%}")

## Recap
- Freeze the eval set **first**; choose checkpoints on a separate **dev** split.
- Report every accuracy with a **CI**, and compare models **paired**, on the same examples.
- Quantization is a trade: measure **accuracy and size (and speed)** together, and decide with a rule you wrote in advance.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def correct_vector(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(-1) == y).numpy().astype(float)

def bootstrap_ci(correct, n_boot=10_000, seed=0):
    correct = np.asarray(correct, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(correct), size=(n_boot, len(correct)))
    means = correct[idx].mean(axis=1)
    return correct.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def paired_bootstrap(correct_a, correct_b, n_boot=10_000, seed=0):
    a, b = np.asarray(correct_a, float), np.asarray(correct_b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n_boot, len(a)))
    d = (b[idx] - a[idx]).mean(axis=1)
    return (b - a).mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def add_lora(model, r=4, alpha=8):
    model = copy.deepcopy(model)
    for p in model.parameters():
        p.requires_grad_(False)
    for i, layer in enumerate(model):
        if isinstance(layer, nn.Linear):
            model[i] = LoRALinear(layer, r, alpha)
    return model

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def quantize_int8(W):
    scale = W.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / 127
    q = torch.round(W / scale).clamp(-127, 127).to(torch.int8)
    return q, scale.float()

def dequantize(q, scale):
    return q.float() * scale